# Streaming turn detection training (Colab)
Select **Runtime → Change runtime type → T4 GPU**, then run every cell in order. Re-running safely resumes the cached baseline.

In [ ]:
# Replace after creating your GitHub repository.
REPO_URL = 'https://github.com/YOUR_GITHUB_USERNAME/streaming-turn-detection.git'
!git clone $REPO_URL turn-detection || (cd turn-detection && git pull)
%cd /content/turn-detection
!pip install -q -r requirements.txt
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ARTIFACTS = Path('/content/drive/MyDrive/turn_detection_artifacts')
for child in ('raw', 'processed', 'cache', 'checkpoints'):
    (ARTIFACTS / child).mkdir(parents=True, exist_ok=True)
print(ARTIFACTS)

In [ ]:
# Start with 5,000 clips to validate quota and throughput. Set MAX_CLIPS=None for a full run.
MAX_CLIPS = 5000
limit = '' if MAX_CLIPS is None else f'--max-pipecat-clips {MAX_CLIPS}'
!python data/prepare_data.py --raw-dir "$ARTIFACTS/raw" --output-dir "$ARTIFACTS/processed" $limit
!cat "$ARTIFACTS/processed/filter_report.json"

In [ ]:
# Existing .pt files are reused; this is safe to rerun after a Colab reset.
!python data/cache_embeddings.py --manifest-dir "$ARTIFACTS/processed" --cache-dir "$ARTIFACTS/cache" --batch-size 32

In [ ]:
# Fresh or resumed training. Save every 250 steps so a Colab disconnect resumes at the next unseen batch.
!python train.py --cache-dir "$ARTIFACTS/cache" --checkpoint-dir "$ARTIFACTS/checkpoints/frozen_bce" --epochs 12 --batch-size 32 --checkpoint-every-steps 250
!cat "$ARTIFACTS/checkpoints/frozen_bce/test_metrics.json"

## Resume after a disconnect
Mount Drive again, rerun the clone/install cell if required, then rerun the cache and training cells. To force an explicit resume:
```bash
python train.py --cache-dir "$ARTIFACTS/cache" --checkpoint-dir "$ARTIFACTS/checkpoints/frozen_bce" --resume "$ARTIFACTS/checkpoints/frozen_bce/last.pt" --epochs 12 --checkpoint-every-steps 250
```